In [2]:
# load dataset
import pandas as pd
import numpy as np

df = pd.read_csv(r"data\pbm_claims_full.csv")

print(df.shape)

(50000, 28)


In [13]:
# convert to datetime
df["fill_date"] = pd.to_datetime(df["fill_date"])

## How many claims were filled each month?

In [15]:
monthly_claims = (
    df.groupby(pd.Grouper(key="fill_date", freq="ME"))["claim_id"]
    .count()
    .reset_index(name="claims_filled")
)

print(monthly_claims)

    fill_date  claims_filled
0  2025-03-31           3959
1  2025-04-30           4068
2  2025-05-31           4263
3  2025-06-30           4133
4  2025-07-31           4261
5  2025-08-31           4269
6  2025-09-30           4099
7  2025-10-31           4218
8  2025-11-30           4126
9  2025-12-31           4175
10 2026-01-31           4350
11 2026-02-28           3826
12 2026-03-31            253


Correct

## How has the usage of Erlotinib changed over time?

In [17]:
erlotinib_trend = (
    df[df["drug_name"] == "Erlotinib"]
    .groupby(pd.Grouper(key="fill_date", freq="ME"))["claim_id"]
    .count()
    .reset_index(name="erlotinib_claims")
)

print(erlotinib_trend)

    fill_date  erlotinib_claims
0  2025-03-31              1339
1  2025-04-30              1399
2  2025-05-31              1390
3  2025-06-30              1400
4  2025-07-31              1417
5  2025-08-31              1368
6  2025-09-30              1360
7  2025-10-31              1390
8  2025-11-30              1357
9  2025-12-31              1389
10 2026-01-31              1468
11 2026-02-28              1277
12 2026-03-31                88


Correct

## Can you show the total rebate for each region?

In [3]:

df.groupby("region")["rebate_estimate"].sum()

region
Midwest      54146239.21
Northeast    54701385.29
South        54316719.21
West         54521945.47
Name: rebate_estimate, dtype: float64

Correct

## Can you show the total copay amount for each drug in each region?

In [4]:
# copay per region per drug
copay_summary = (
    df.groupby(["region", "drug_name"], as_index=False)["copay"]
    .sum()
)

print(copay_summary)

       region    drug_name       copay
0     Midwest    Erlotinib   999147.33
1     Midwest    Gefitinib   980056.58
2     Midwest  Osimertinib  1013316.18
3   Northeast    Erlotinib  1028211.06
4   Northeast    Gefitinib  1028227.70
5   Northeast  Osimertinib  1039321.12
6       South    Erlotinib  1033657.98
7       South    Gefitinib  1013525.50
8       South  Osimertinib  1043143.44
9        West    Erlotinib  1002035.25
10       West    Gefitinib  1010150.35
11       West  Osimertinib  1013427.13


correct

## What is the total pharmacy spending in each region?

In [5]:
df["total_cost"] = df["ingredient_cost"] + df["dispensing_fee"]

spending_by_region = (
    df.groupby("region", as_index=False)["total_cost"]
    .sum()
)

# scientific notation to full numbers
pd.options.display.float_format = '{:,.0f}'.format
print(spending_by_region)

      region  total_cost
0    Midwest 241,139,140
1  Northeast 243,982,176
2      South 242,997,042
3       West 240,676,499


correct

## Which drugs generate the highest total cost for the health plan?

In [6]:
drug_costs = (
    df.groupby("drug_name", as_index=False)["plan_paid_amount"]
    .sum()
    .sort_values(by="plan_paid_amount", ascending=False)
)

print(drug_costs)

     drug_name  plan_paid_amount
2  Osimertinib       295,008,075
1    Gefitinib       231,081,167
0    Erlotinib       212,815,106


correct

## Which therapeutic classes contribute the most to overall drug spending?

In [7]:
class_spending = (
    df.groupby("therapeutic_class", as_index=False)["ingredient_cost"]
    .sum()
    .sort_values(by="ingredient_cost", ascending=False)
)

print(class_spending)

  therapeutic_class  ingredient_cost
0          EGFR TKI      966,802,354


correct

## Which pharmacy types process the most prescriptions?

In [8]:
pharmacy_claims = (
    df.groupby("pharmacy_type")
    .size()
    .reset_index(name="claim_count")
    .sort_values(by="claim_count", ascending=False)
)

print(pharmacy_claims)

         pharmacy_type  claim_count
2               Retail        22618
3            Specialty        17399
1           Mail Order         7542
0  Hospital Outpatient         2441


partial Wrong - pharmacy_type correct but claim_count not shown